# 🔍 Vietnamese Fake News Detection System - Colab Test Notebook

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab**. Nó sẽ tự động tải mã nguồn từ GitHub, cài đặt môi trường và chạy thử nghiệm hệ thống.

**Tác giả:** UET Project Team

## 📥 Bước 1: Clone Repository từ GitHub

Tải toàn bộ source code của dự án về môi trường Colab và di chuyển vào thư mục dự án.

In [1]:
# Xóa thư mục cũ nếu đã clone trước đó để tránh lỗi
!rm -rf fake-news-detection

# Clone repository (thay link bằng link thực tế nếu cần)
!git clone https://github.com/phidinhmanh/fake-news-detection.git

# Di chuyển thư mục làm việc vào trong project
%cd fake-news-detection

Cloning into 'fake-news-detection'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 234 (delta 76), reused 208 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (234/234), 576.42 KiB | 2.88 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/fake-news-detection


## 🛠️ Bước 2: Cài đặt Dependencies

Cài đặt các thư viện xử lý tiếng Việt (`underthesea`), NLP (`transformers`), Vector Database (`lancedb`), và các công cụ khác.

In [2]:
!pip install -q underthesea transformers datasets lancedb sentence-transformers google-generativeai langdetect wordcloud plotly streamlit pyngrok python-dotenv huggingface_hub

## ⚙️ Bước 3: Thiết lập API Keys (Sử dụng getpass)

Nhập các API Key một cách bảo mật ngay trong notebook Colab.

In [3]:
import os
import sys
import getpass
from pathlib import Path
from huggingface_hub import login

# Xác định thư mục gốc project
ROOT_DIR = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.append(str(ROOT_DIR))
print(f"Project Root: {ROOT_DIR}")

# ── Google Gemini API Key ────────────────────────────────────────────────
def setup_google_api():
    key = os.getenv("GOOGLE_API_KEY")
    if not key:
        # Thử lấy từ Colab Secrets trước
        try:
            from google.colab import userdata
            key = userdata.get('GOOGLE_API_KEY')
        except:
            pass
    
    if not key:
        print("🔑 Vui lòng nhập GOOGLE_API_KEY (Gemini API):")
        key = getpass.getpass()
    
    if key:
        os.environ["GOOGLE_API_KEY"] = key
        print("✅ GOOGLE_API_KEY đã được thiết lập.")
    else:
        print("⚠️ Không có GOOGLE_API_KEY - Agent sẽ chạy ở chế độ MOCK.")

# ── Hugging Face Token ────────────────────────────────────────────
def setup_hf_token():
    token = os.getenv("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except:
            pass
            
    if not token:
        print("🔑 Vui lòng nhập HF_TOKEN (Hugging Face Hub):")
        token = getpass.getpass()
    
    if token:
        os.environ["HF_TOKEN"] = token
        login(token=token)
        print("✅ Đã đăng nhập Hugging Face Hub.")
    else:
        print("⚠️ Không có HF_TOKEN - Việc tải model có thể bị giới hạn.")

setup_google_api()
setup_hf_token()

Project Root: /content/fake-news-detection
🔑 Vui lòng nhập GOOGLE_API_KEY (Gemini API):
⚠️ Không có GOOGLE_API_KEY - Agent sẽ chạy ở chế độ MOCK.
🔑 Vui lòng nhập HF_TOKEN (Hugging Face Hub):


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


HfHubHTTPError: Invalid user token. The token from HF_TOKEN environment variable is invalid. Note that HF_TOKEN takes precedence over `hf auth login`.

## 📂 Bước 4: Chuẩn bị dữ liệu (Mock data)

Tạo dữ liệu giả lập (đủ lớn) để test ngay pipeline xử lý mà không bị lỗi chia tập dữ liệu.

In [ ]:
import config
import pandas as pd
import numpy as np

config.DATASET_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.KB_DIR.mkdir(parents=True, exist_ok=True)

# Tạo 20 dòng dữ liệu để tránh lỗi train_test_split (stratify)
texts = [
    "Vaccine COVID-19 cực kỳ nguy hiểm, hãy chia sẻ ngay!",
    "Thủ tướng họp về phát triển kinh tế xã hội quý 1.",
    "Giá xăng tăng kỷ lục vào ngày mai.",
    "5.000 người đã tử vong vì ăn nhầm loại rau này.",
    "Chính phủ ban hành quy định mới về thuế thu nhập cá nhân.",
    "Phát hiện người ngoài hành tinh tại Việt Nam.",
    "Đội tuyển bơi lội giành 10 HCV tại giải vô địch.",
    "Uống nước chanh pha mật ong đuổi sạch virus cúm.",
    "Lạm phát tháng 3 giảm sát mức 0.",
    "Trái đất sẽ ngừng quay trong 5 ngày tới.",
    "Bí quyết chữa ung thư bằng lá đu đủ sau 3 ngày.",
    "Ngân hàng Nhà nước điều chỉnh lãi suất điều hành.",
    "Sập cầu Cần Thơ là tin đồn nhảm nhí.",
    "Việt Nam xuất khẩu gạo đứng đầu thế giới năm qua.",
    "Thuốc tiên giúp trẻ mãi không già đã xuất hiện.",
    "Hội nghị thượng đỉnh về biến đổi khí hậu diễn ra tại Hà Nội.",
    "Ăn tỏi sống có thể tiêu diệt hoàn toàn tế bào ung thư.",
    "Bộ Giáo dục công bố phương án thi tốt nghiệp THPT mới.",
    "Sóng thần sắp đổ bộ vào bờ biển miền Trung.",
    "Tăng trưởng GDP năm nay dự báo đạt mức 6.5%."
]

labels = ["fake", "real", "fake", "fake", "real", "fake", "real", "fake", "real", "fake"] * 2
sources = ["social", "news", "news", "social", "news", "social", "news", "social", "news", "social"] * 2

mock_data = pd.DataFrame({
    'text': texts,
    'label': labels,
    'source': sources,
    'lang': ['vi'] * 20
})

mock_path = config.DATASET_PROCESSED_DIR / "preprocessed_all.csv"
mock_data.to_csv(mock_path, index=False)
print(f"✅ Đã tạo mock data tại {mock_path} với {len(mock_data)} dòng.")

## 📊 Bước 5: Chạy Pipeline Xử lý dữ liệu

Gộp dữ liệu và trích xuất đặc trưng phong cách viết.

In [ ]:
# Chạy gộp dữ liệu (sẽ dùng mock data vừa tạo)
!python dataset/download_datasets.py
!python dataset/preprocess_vietnamese.py
!python dataset/merge_datasets.py

# Trích xuất features
!python dataset/feature_extraction.py

## 🤖 Bước 6: Test LLM Agent Pipeline

Chạy thử pipeline 7 giai đoạn (Sequential Adversarial) để kiểm chứng tính năng agentic.

In [ ]:
import os
from sequential_adversarial.pipeline import SequentialAdversarialPipeline

# Kiểm tra xem có API Key không để quyết định chế độ chạy
api_key = os.getenv("GOOGLE_API_KEY")
use_mock = not api_key

if use_mock:
    print("ℹ️ Chạy ở chế độ MOCK (do thiếu API Key)")
else:
    print("🚀 Chạy ở chế độ REAL (Gemini API)")

# Fix: Sử dụng tham số 'mock' thay vì 'mock_fallback'
pipeline = SequentialAdversarialPipeline(mock=use_mock)

test_article = """
KHẨN CẤP: Vaccine COVID-19 gây ra hàng nghìn ca tử vong trên toàn thế giới! 
Theo một nghiên cứu bí mật bị rò rỉ, các hãng dược phẩm đã che giấu sự thật 
về tác dụng phụ nghiêm trọng của vaccine. Hãy chia sẻ thông tin này ngay!!!
"""

print("⏳ Đang khởi động Pipeline & Phân tích bài viết...")
try:
    result = pipeline.run(test_article)
    
    print("\n" + "="*50)
    print("📊 KẾT QUẢ TỪ AGENTIC PIPELINE")
    print("="*50)

    if result.verity_report:
        print(f"🔥 KẾT LUẬN: {result.verity_report.conclusion.upper()}")
        print(f"🎯 Độ tin cậy: {result.verity_report.confidence:.0%}")
        print(f"📝 Tóm tắt bằng chứng: {result.verity_report.evidence_summary}")
        print(f"🧐 Phân tích định kiến: {result.verity_report.bias_summary}")
        print("\n🔍 Các phát hiện chính:")
        for f in result.verity_report.key_findings:
            print(f"  - {f}")
    else:
        print("❌ Lỗi: Pipeline không sinh được báo cáo cuối cùng.")

    print(f"\n📢 Trích xuất được {len(result.claims)} luận điểm.")
    
    if result.tfidf_comparison:
        print("\n🤖 Đối chiếu Baseline (TF-IDF):")
        print(f"   - Nhãn dự đoán: {result.tfidf_comparison.tfidf_label}")
        print(f"   - Độ tin cậy: {result.tfidf_comparison.tfidf_confidence:.2f}")
        print(f"   - Thống nhất với LLM: {'CÓ' if result.tfidf_comparison.agreement else 'KHÔNG'}")
        
    print("\n✅ Pipeline hoàn thành xuất sắc.")
    
except Exception as e:
    print(f"❌ Lỗi thực thi Pipeline: {e}")
    import traceback
    traceback.print_exc()

## 🧠 Bước 7: Thử nghiệm PhoBERT Tokenizer

Đảm bảo tải được mô hình `vinai/phobert-base` từ Hugging Face.

In [ ]:
import torch
from transformers import AutoTokenizer
from model.phobert_baseline import PhoBERTBaseline

try:
    tokenizer = PhoBERTBaseline.get_tokenizer()
    print("✅ Tải PhoBERT Tokenizer thành công!")
    
    # Thử tokenize 1 câu tiếng Việt
    inputs = tokenizer("Tôi là sinh viên đại học Công Nghệ", return_tensors="pt")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
except Exception as e:
    print(f"❌ Lỗi: {e}")

---

# 🚀 Thực thi quy trình nâng cao (Tùy chọn)

## 📥 Bước 8: Tải & Tiền xử lý dữ liệu thật

Tải ViFactCheck từ HuggingFace (Yêu cầu `HF_TOKEN` nếu dataset private/restricted).

In [ ]:
!python dataset/download_datasets.py
!python dataset/preprocess_vietnamese.py
!python dataset/merge_datasets.py
!python dataset/feature_extraction.py

## 🎯 Bước 9: Huấn luyện PhoBERT (Cần GPU)

Vui lòng đổi Runtime sang **T4 GPU**.

In [ ]:
import os
from config import DATASET_PROCESSED_DIR

# Kiểm tra file train.csv trước khi train để tránh lỗi
train_path = DATASET_PROCESSED_DIR / 'train.csv'
if not train_path.exists():
    print('⚠️ Không tìm thấy train.csv! Đang tạo nhanh mock data...')
    # Chạy lại bước merge_datasets nếu thiếu
    !python dataset/merge_datasets.py

if train_path.exists():
    !python model/train_phobert.py --variant baseline --epochs 1
else:
    print('❌ Lỗi: Vẫn không tìm thấy dữ liệu để huấn luyện. Vui lòng kiểm tra Bước 4 & 5.')

## 🌐 Bước 10: Chạy Demo Streamlit (LocalTunnel)

Mở URL `loca.lt` được sinh ra để xem giao diện web.

In [ ]:
!npm install -g localtunnel
print("Mật khẩu LocalTunnel của bạn là IP xuất hiện dưới đây:")
!wget -q -O - ipv4.icanhazip.com
print("\n🚀 Đang khởi động Streamlit...")
!streamlit run ui/app.py & npx localtunnel --port 8501